# Clark, Abbott & Sompolinsky (2026): the tuning-curve claims, on DANDI:000939

Paper. D. G. Clark, L. F. Abbott, H. Sompolinsky, "Symmetries and continuous attractors in
disordered neural circuits", Nature Neuroscience (2026), doi:10.1038/s41593-026-02452-4. The
claims file `claims/clark-2026.yaml` was written from the version of record (main text, Methods,
figure legends and the Supplementary Information).

Dataset. DANDI:000939 (Duszkiewicz et al. 2024, postsubiculum of freely moving mice), pinned
at version 0.240528.1542 (doi:10.48324/dandi.000939/0.240528.1542). This is the version the
paper cites (claims file, opening comment). Later versions exist (0.250207.0025 to
0.261016.2310); the cell below shows that every file changed after this version (new names with
session ids and sizes of about 24 GB), so the cited version is the one that holds the files the
paper's code read. The paper reuses this dataset (`relation: reuses`).

Outside input. The paper's cached tuning curves, `mouse_data/tc_data.npz` in Zenodo
doi:10.5281/zenodo.21827768 (version 1.0), read by range requests from the record's tar and
checked against the record's MD5. It is used only by the link claim `tuning-curves-match-cache`.
Everything else is computed from the NWB files, streamed from the archive.

Code followed. github.com/davidclark1/symmetries-continuous-attractors at commit 245726f
(the claims file cites it); `hdring.py` in this entry repeats the parts used, in numpy, and names
the function of the paper's code each one repeats.

This notebook tests the claims on the tuning curves and the generative process. The claims on
the data-derived network are in `network.ipynb`.

| claim | tested here |
|---|---|
| hd-cell-selection (link) | yes |
| tuning-curves-match-cache (link, with inputs) | yes |
| fig1e-heterogeneous-peaks | yes |
| fig5d-peak-numbers | yes |
| fig1fg-mean-and-fluctuation-profiles | yes |
| fig3ab-centers-of-mass-uniform | yes |
| fig3c-correlation-approaches-circulant | yes |
| fig4-generative-fit | yes |
| fig5ab-profiles-match | yes |
| fig5c-peak-counts-match | yes |
| fig5d-peak-heights-match | yes |
| fig5e-flip-symmetry-match | yes |
| fig5f-information-match | yes |
| fig2l-recorded-bump-tracks-hd | yes |
| network claims (fig2b to figs14) | in network.ipynb |
| simulation-only claims (data: elsewhere) | see the attempt record |

In [ ]:
import os, time, resource
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from astropy.stats import kuiper, circcorrcoef
from statsmodels.stats.multitest import multipletests
import hdring as H

T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", H.DANDISET, "version", H.VERSION)

## Files and versions

In [ ]:
assets = H.list_assets()
order = H.session_order(assets)
print(len(assets), "assets at", H.VERSION)
for a in sorted(assets, key=lambda a: a["path"]):
    print(f"  {a['path']:50s} {a['size']/1e9:6.2f} GB")
# compare with the latest published version
vers = requests.get(f"{H.API}/dandisets/{H.DANDISET}/versions/?page_size=100").json()["results"]
pub = [v["version"] for v in vers if v["version"] != "draft"]
latest = sorted(pub)[-1]
la = H.list_assets(latest)
blobs_pinned = {a["blob"] for a in assets}
print("published versions:", ", ".join(sorted(pub)))
print(f"latest {latest}: {len(la)} assets, sizes {min(a['size'] for a in la)/1e9:.1f} to {max(a['size'] for a in la)/1e9:.1f} GB;",
      "files shared with the pinned version (same blob):", sum(a["blob"] in blobs_pinned for a in la))
print("example path in latest:", la[0]["path"])

## Load every session

One file per subject (the `+ogen.nwb` file where a session has one, as `ring/data.py`
`get_session_info` chooses; at this version each subject has exactly one file), in the code's
sorted order, which fixes the session index. For each file the epochs, the head direction and
the units table are read; spike times are read for the head-direction units only. The
cross-validated tuning curves are computed as `ring/data.py` `process_file` does (helper,
`tuning_curves_session`).

In [ ]:
sessions = []
t = time.time()
for i, (path, aid, subj, ogen) in enumerate(order):
    s = H.load_session(aid)
    plain, opt, sig = H.tuning_curves_session(s)
    hd_idx, ti, tf, ti_idx, tf_idx = H.epoch_bins(s)
    sessions.append(dict(idx=i, subject=subj, path=path, ogen=ogen, sess=s, plain=plain, opt=opt, sig=sig,
                         hd_idx=hd_idx, ti=ti, tf=tf))
print(f"loaded {len(sessions)} sessions in {time.time()-t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

In [ ]:
rows = []
for d in sessions:
    s = d["sess"]
    hd = s["hd"]
    dt = np.diff(s["hd_t"])
    ep = {t: (a, b) for t, a, b in s["epochs"]}
    rows.append(dict(
        session=d["idx"], subject=d["subject"], file_ogen=d["ogen"], units=s["n_units"],
        hd_units=int(s["is_hd"].sum()), exc_units=int(s["is_exc"].sum()), fs_units=int(s["is_fs"].sum()),
        hd_and_exc=int((s["is_hd"] & s["is_exc"]).sum()), hd_not_exc=int((s["is_hd"] & ~s["is_exc"]).sum()),
        epochs="/".join(t for t, _, _ in s["epochs"]),
        square_s=round(d["tf"] - d["ti"], 1), hd_dt_median_ms=round(np.median(dt) * 1e3, 3),
        hd_dt_max_in_square_ms=round(np.max(np.diff(s["hd_t"][(s["hd_t"] >= d["ti"]) & (s["hd_t"] <= d["tf"])])) * 1e3, 1),
        hd_min=round(np.nanmin(hd), 4), hd_max=round(np.nanmax(hd), 4), hd_nan_square=round((d["hd_idx"] < 0).mean(), 4),
        spikes_in_square=int(sum(((x >= d["ti"]) & (x < d["tf"])).sum() for x in s["spikes"])),
        unsorted_spike_trains=int(sum(np.any(np.diff(x) < 0) for x in s["spikes"])),
    ))
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
print("head-direction unit:", sessions[0]["sess"]["hd_unit"])
print("totals: sessions", len(tab), "| units", tab.units.sum(), "| head-direction units", tab.hd_units.sum(),
      "| of which excitatory", tab.hd_and_exc.sum(), "| sessions with a +ogen file", int(tab.file_ogen.sum()),
      "| sessions with a wake_opto epoch", int(tab.epochs.str.contains("wake_opto").sum()))
print("one subject per session:", tab.subject.nunique() == len(tab))
print("bins of the 100-bin head-direction grid never visited in a square epoch, per session (max):",
      max(int((np.bincount(d['hd_idx'][d['hd_idx'] >= 0], minlength=100) == 0).sum()) for d in sessions))

### Recordings deposited more than once, and the shared clock

Duplicates: the number of head-direction samples and the first spike time of the first unit are
compared across sessions. Clock: for every session, the head-direction information of the plain
(unsmoothed) tuning curves of its head-direction units over the square epoch is computed with
the spike times shifted by a lag between -2 s and +2 s. If spikes and head direction share a
clock, the information peaks near lag 0 (a head-direction cell's tuning is sharpest at a lag of
a few tens of ms, its anticipatory time interval).

In [ ]:
sig_keys = [(len(d["sess"]["hd"]), round(float(d["sess"]["spikes"][0][0]), 6) if len(d["sess"]["spikes"][0]) else None) for d in sessions]
dups = [(sessions[i]["subject"], sessions[j]["subject"]) for i in range(len(sessions)) for j in range(i + 1, len(sessions)) if sig_keys[i] == sig_keys[j]]
print("sessions with identical head-direction sample count and first spike time:", dups if dups else "none")
nhd = [len(d["sess"]["hd"]) for d in sessions]
print("distinct head-direction sample counts:", len(set(nhd)), "of", len(nhd))

lags = np.array([-2, -1, -0.5, -0.2, -0.1, -0.05, -0.02, 0, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2])
def info_at_lag(d, lag):
    hd_idx, ti = d["hd_idx"], d["ti"]
    N = len(hd_idx); ok = hd_idx >= 0
    occ = np.bincount(hd_idx[ok], minlength=100).astype(float)
    vals = []
    for st in d["sess"]["spikes"]:
        c = H.spike_counts(st + lag, ti, d["tf"], N)
        r = np.bincount(hd_idx[ok], weights=c[ok], minlength=100) / np.maximum(occ, 1)
        p = occ / occ.sum(); lam = (p * r).sum()
        if lam <= 0: continue
        q = r / lam
        vals.append(np.nansum(p * q * np.log2(np.where(q > 0, q, 1))))
    return np.mean(vals)
clock = np.array([[info_at_lag(d, L) for L in lags] for d in sessions])
best = lags[clock.argmax(1)]
ct = pd.DataFrame(clock.round(3), columns=[f"{L:+g}s" for L in lags]); ct.insert(0, "subject", [d["subject"] for d in sessions])
print("mean information (bits per spike) of the head-direction units, spikes shifted by the lag:")
print(ct.to_string(index=False))
print("lag of maximum information per session:", dict(zip([d['subject'] for d in sessions], best)))
print("sessions whose maximum is within 0.1 s of lag 0:", int((np.abs(best) <= 0.1).sum()), "of", len(best))
print("ratio of information at lag 0 to that at +-2 s (median over sessions):",
      round(float(np.median(clock[:, lags == 0][:, 0] / np.maximum(clock[:, 0], clock[:, -1]))), 2))

## hd-cell-selection (link)

> The dandiset holds the 31 sessions, one per mouse, with 2,691 units, of which 1,533 carry the
> head-direction label; nine of the mice, which also underwent optogenetic manipulation in a
> separate epoch, contribute 315 of those cells; the four mice with most head-direction cells
> have 117, 93, 90 and 86.

In [ ]:
units = tab.units.sum()
hd_a = tab.hd_units.sum()                     # reading (a), is_head_direction
hd_b = tab.hd_and_exc.sum()                   # reading (b), is_head_direction and is_excitatory
opt_mean = [d["opt"].mean(1) for d in sessions]
nz = np.concatenate([~np.all(o == 0, axis=1) for o in opt_mean])
opto = tab[tab.epochs.str.contains("wake_opto")]
print("sessions (mice):", len(tab), "(expected 31)")
print("units in all sessions:", units, "(expected 2691)")
print("head-direction units, reading (a) is_head_direction:", hd_a, "(expected 1533)")
print("head-direction units, reading (b) is_head_direction and is_excitatory:", hd_b)
print("reading (a) after the code's drop of all-zero final curves (nz_mask):", int(nz.sum()), "dropped:", int((~nz).sum()))
print("mice with a wake_opto epoch:", len(opto), list(opto.subject), "; their head-direction cells:", opto.hd_units.sum(), "(expected 315)")
top = tab.sort_values("hd_units", ascending=False).head(4)
print("four mice with most head-direction cells:", list(zip(top.subject, top.hd_units)), "(expected A3707 117, A3716 93, A3711 90, A3706 86)")
print("head-direction cells of A3707:", int(tab.set_index('subject').loc['A3707', 'hd_units']), "(expected 117)")

**Result: reproduced.** The files hold 31 sessions, one per subject, with 2,691 units, of which
1,533 carry `is_head_direction`; all 1,533 are also `is_excitatory`, so readings (a) and (b) give
the same count, and the code's drop of all-zero curves removes none. The nine mice with a
`wake_opto` epoch hold 315 head-direction cells, and the four largest sessions are A3707 (117),
A3716 (93), A3711 (90) and A3706 (86).

## tuning-curves-match-cache (link, with the Zenodo input)

> Tuning curves computed from the dandiset with the paper's procedure agree with the tuning
> curves in the paper's cached results on Zenodo.

In [ ]:
b, md5_calc, md5_listed = H.read_zenodo_member("mouse_data/tc_data.npz")
print("tc_data.npz MD5", md5_calc, "listed", md5_listed, "match:", md5_calc == md5_listed)
cache = H.load_npz_bytes(b)
c_names = cache["mouse_names"]; c_nums = cache["mouse_nums"]; c_hd = cache["hd_mask"]
print("cache: units", len(c_hd), "head-direction", int(c_hd.sum()), "sessions", len(cache["unit_counts"]))
print("cache session order equals ours:", [str(n) for n in pd.unique(c_names)] == [d["subject"] for d in sessions])
print("cache unit counts equal ours:", list(cache["unit_counts"]) == list(tab.units))
print("cache hd labels equal ours:", np.array_equal(c_hd, np.concatenate([d["sess"]["is_hd"] for d in sessions])),
      "| exc labels:", np.array_equal(cache["exc_mask"], np.concatenate([d["sess"]["is_exc"] for d in sessions])),
      "| fs labels:", np.array_equal(cache["fs_mask"], np.concatenate([d["sess"]["is_fs"] for d in sessions])))
ours_opt = np.concatenate([d["opt"] for d in sessions]); ours_sig = np.concatenate([d["sig"] for d in sessions])
ours_plain = np.concatenate([d["plain"] for d in sessions])
c_opt = cache["opt_tcs"][c_hd]; c_sig = cache["opt_sigmas"][c_hd]; c_plain = cache["plain_tcs"][c_hd]
r = np.array([np.corrcoef(a.mean(0), b_.mean(0))[0, 1] for a, b_ in zip(ours_opt, c_opt)])
print("units compared:", len(r))
print(f"Pearson r of partition-averaged curves: min {r.min():.6f}, median {np.median(r):.6f}; fraction >= 0.99: {(r >= 0.99).mean():.4f}")
print("max abs difference of partition-averaged curves (Hz):", float(np.abs(ours_opt.mean(1) - c_opt.mean(1)).max()))
print("max abs difference of plain curves (Hz):", float(np.abs(ours_plain - c_plain).max()))
same_sig = ours_sig == c_sig
print(f"chosen widths equal for both partitions: {same_sig.all(1).mean():.4f} of units; per partition: {same_sig.mean():.4f}")
diff_curve = np.abs(ours_opt - c_opt).max(2)
print("where the widths differ, max abs curve difference:", float(diff_curve[~same_sig].max()) if (~same_sig).any() else None,
      "| our width range there:", (float(ours_sig[~same_sig].min()), float(ours_sig[~same_sig].max())) if (~same_sig).any() else None,
      "| cache width range there:", (float(c_sig[~same_sig].min()), float(c_sig[~same_sig].max())) if (~same_sig).any() else None)

**Result: reproduced.** The record's MD5 matches, and the cache has the same session order, unit
counts and labels as the files. All 1,533 recomputed partition-averaged curves correlate at r =
1.000000 with the cached ones (largest absolute difference 1.4e-14 Hz). The chosen widths agree
for both partitions in 78% of units; where they differ the curves still agree to 2e-14 Hz, so
those widths give the same smoothed curve (ties).

## The curves used below

As `data_and_generative_model.ipynb` cell 3: the partition-averaged curves of the units with
`is_head_direction` and a non-zero curve, each divided by its mean over the 100 bins.

In [ ]:
hdmask_all = np.concatenate([d["sess"]["is_hd"][d["sess"]["spike_units"]] for d in sessions])  # all True (hd units loaded)
Phi_list, mouse_nums = [], []
for d in sessions:
    o = d["opt"].mean(1)
    keep = ~np.all(o == 0, axis=1)
    o = o[keep]
    Phi_list.append(o / o.mean(1, keepdims=True)); mouse_nums += [d["idx"]] * len(o)
Phi_data = np.concatenate(Phi_list); mouse_nums = np.array(mouse_nums)
names = np.array([d["subject"] for d in sessions])
print("Phi_data", Phi_data.shape, "range", Phi_data.min().round(4), Phi_data.max().round(3), "row means", Phi_data.mean(1).min().round(6), Phi_data.mean(1).max().round(6))

## fig1e-heterogeneous-peaks

> Mouse head-direction cells are heterogeneous, with variable numbers of irregularly spaced
> peaks, unlike the classical prediction of tuning curves identical up to shifts.

In [ ]:
pc1 = H.peak_counts(Phi_data, 1.0)
print("peak-count distribution at zthresh 1:", dict(zip(*np.unique(pc1, return_counts=True))))
seps = []
for tcv in Phi_data[pc1 >= 2]:
    mx = H.find_local_maxima(tcv, 1.0)
    top2 = mx[np.argsort(tcv[mx])[::-1][:2]]
    seps.append(H.circ_diff(*(2 * np.pi * top2 / 100)))
seps = np.array(seps)
q = np.percentile(seps, [0, 10, 25, 50, 75, 90, 100])
print("multi-peaked curves:", len(seps))
print("separation of the two highest peaks (rad), percentiles 0,10,25,50,75,90,100:", q.round(3))
print(f"interquartile range {q[4]-q[2]:.3f} rad against pi/4 = {np.pi/4:.3f}")
print("histogram of separations (8 bins over 0 to pi):", np.histogram(seps, bins=8, range=(0, np.pi))[0])
for z in (0.5, 0.0):
    pcz = H.peak_counts(Phi_data, z)
    print(f"variant, zthresh {z}: peak-count distribution", dict(zip(*np.unique(pcz, return_counts=True))))
by_mouse = pd.DataFrame(dict(subject=names[mouse_nums], multi=pc1 >= 2)).groupby("subject").multi.agg(["sum", "count"])
print("curves with 2 or more peaks by mouse (sum of count):", by_mouse.apply(lambda r: f"{r['sum']}/{r['count']}", axis=1).to_dict())

**Result: reproduced.** Peak counts at zthresh 1 range from 1 to 6 (1,121 curves with one peak,
357 with two, 48 with three). The two highest peaks of the 412 multi-peaked curves are separated
by 0.13 to 3.14 rad, with an interquartile range of 1.38 rad, above pi/4 (0.79).

## fig5d-peak-numbers

> At zthresh 1, all 1,533 experimental curves have at least one peak, 412 have at least two and
> 55 at least three; of 100,000 generative-process samples, 24,271 have at least two and 1,043 at
> least three.

The generative samples are drawn at the paper's fitted parameters, as the criterion says, from
this claim's seed (protocol); the paper's notebook used numpy seed 42 with the legacy global
generator, whose draws a seeded `Generator` does not repeat. The tolerance covers this. The
result is also given under four more seeds.

In [ ]:
PAPER_PARAMS = (1.4211, 1.7307, 2.7659)   # scale, bias, beta (claims file; notebook printout)
print("data, at least 1, 2, 3 peaks:", [(pc1 >= k).sum() for k in (1, 2, 3)], "(expected 1533, 412, 55)")
gen_counts = {}
for k in range(5):
    sd = H.seed("fig5d-peak-numbers") + 1000 * k
    G = H.gen_Phi_m(*PAPER_PARAMS, 100000, np.random.default_rng(sd))
    pg = H.peak_counts(G, 1.0)
    gen_counts[sd] = [(pg >= j).sum() for j in (1, 2, 3)]
    print(f"seed {sd}: generative, at least 1, 2, 3 peaks of 100,000:", gen_counts[sd], "(expected 100000, 24271, 1043)")
v2 = [c[1] for c in gen_counts.values()]; v3 = [c[2] for c in gen_counts.values()]
print("range over five seeds: >=2", min(v2), max(v2), "| >=3", min(v3), max(v3))

**Result: reproduced.** The data give 1,533, 412 and 55 curves with at least 1, 2 and 3 peaks, the
paper's numbers exactly. Of 100,000 generative samples at the paper's parameters, 23,908 have at
least two and 1,052 at least three under the claim's seed (expected 24,271 within 2,427 and 1,043
within 104); over five seeds 23,908 to 24,267 and 1,001 to 1,082, all within tolerance.

## fig1fg-mean-and-fluctuation-profiles

> Aligned by their circular centers of mass within each of the 31 mice, the tuning curves have
> unimodal, symmetric mean profiles, but fluctuations about them comparable in magnitude to the
> means, and both profiles are consistent across mice.

In [ ]:
def mouse_profiles(Phi, mn):
    mus, sds = [], []
    for m in range(mn.max() + 1):
        c = H.center_tuning_curves(Phi[mn == m]); mus.append(c.mean(0)); sds.append(c.std(0))
    return np.array(mus), np.array(sds)
mus, sds = mouse_profiles(Phi_data, mouse_nums)
mu_avg, sd_avg = mus.mean(0), sds.mean(0)
refl = lambda p: np.roll(p[::-1], 1)
maxima = np.where((mu_avg > np.roll(mu_avg, 1)) & (mu_avg > np.roll(mu_avg, -1)))[0]
off = lambda i: (i + 50) % 100 - 50
print("mouse-average mean profile: local maxima at offsets", [off(i) for i in maxima], "| peak", mu_avg.max().round(3), "at offset", off(mu_avg.argmax()))
unimodal = len(maxima) == 1 and abs(off(maxima[0])) <= 2
r_sym = np.corrcoef(mu_avg, refl(mu_avg))[0, 1]
print("unimodal:", unimodal, "| symmetry r with reflection:", round(r_sym, 4), "(>= 0.9)")
ra = 0.5 * mu_avg.max() <= sd_avg.max() <= 2 * mu_avg.max()
rb = sd_avg[0] > 0.5 * mu_avg[0]
print(f"peak of sd profile {sd_avg.max():.3f}, peak of mean profile {mu_avg.max():.3f}, ratio {sd_avg.max()/mu_avg.max():.3f}; reading (a) holds: {ra}")
print(f"at offset 0: sd {sd_avg[0]:.3f}, mean {mu_avg[0]:.3f}, ratio {sd_avg[0]/mu_avg[0]:.3f}; reading (b) holds: {rb}")
r_mu = np.array([np.corrcoef(m, mu_avg)[0, 1] for m in mus]); r_sd = np.array([np.corrcoef(s_, sd_avg)[0, 1] for s_ in sds])
print(f"consistency: median r of mouse mean profiles with the average {np.median(r_mu):.3f} (min {r_mu.min():.3f}); of sd profiles {np.median(r_sd):.3f} (min {r_sd.min():.3f})")
per = pd.DataFrame(dict(subject=names, n=[(mouse_nums == m).sum() for m in range(31)], mean_peak=mus.max(1).round(2), sd_peak=sds.max(1).round(2),
                        sd_over_mean_at0=(sds[:, 0] / mus[:, 0]).round(2), r_mean=r_mu.round(3), r_sd=r_sd.round(3), sym_r=[round(np.corrcoef(m, refl(m))[0, 1], 3) for m in mus]))
print(per.to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(7, 2.3))
x = np.arange(-50, 50) * 2 * np.pi / 100
for m in range(31):
    ax[0].plot(x, np.roll(mus[m], 50), c="0.7", lw=0.5); ax[1].plot(x, np.roll(sds[m], 50), c="0.7", lw=0.5)
ax[0].plot(x, np.roll(mu_avg, 50), c="k"); ax[1].plot(x, np.roll(sd_avg, 50), c="k")
ax[0].set_title("mean profile"); ax[1].set_title("sd profile"); ax[0].set_xlabel("offset (rad)"); ax[1].set_xlabel("offset (rad)")
plt.show()
print("plotted ranges: mean", mus.min().round(3), mus.max().round(3), "| sd", sds.min().round(3), sds.max().round(3))

**Result: reproduced.** The mouse-average mean profile has one local maximum, at offset 0, and
correlates with its reflection at r = 0.9998. The peak of the standard deviation profile is 0.56
times the peak of the mean (2.64 against 4.72), which satisfies both readings of comparable. The
median correlation of a mouse's profile with the average is 0.996 for the mean and 0.974 for the
standard deviation.

## fig3ab-centers-of-mass-uniform

> Within each session the tuning-curve centers of mass are distributed uniformly; the Kuiper test
> rejects uniformity in 2 of 31 sessions before correction and in none after Benjamini-Hochberg
> correction, with corrected P values from 0.59 to 0.97.

In [ ]:
coms = H.circular_center_of_mass(Phi_data)
pv, stat = [], []
for m in range(31):
    D, p = kuiper(coms[mouse_nums == m], lambda x: x / (2 * np.pi)); pv.append(p); stat.append(D)
pv = np.array(pv)
rej, pcor, _, _ = multipletests(pv, alpha=0.05, method="fdr_bh")
print("sessions with uncorrected P < 0.05:", int((pv < 0.05).sum()), "(expected 2)")
print("sessions significant after BH:", int(rej.sum()), "(expected 0)")
print(f"corrected P range {pcor.min():.3f} to {pcor.max():.3f} (expected 0.59 to 0.97); uncorrected {pv.min():.3f} to {pv.max():.3f}")
print(pd.DataFrame(dict(subject=names, n=[(mouse_nums == m).sum() for m in range(31)], kuiper_V=np.round(stat, 3), p=pv.round(4), p_bh=pcor.round(4))).to_string(index=False))

**Result: reproduced.** Two sessions have uncorrected P < 0.05 (A3711, 0.044; A9701, 0.048), none
is significant after Benjamini-Hochberg correction, and the corrected P values range from 0.592 to
0.968, as the paper reports (0.59 to 0.97).

## fig3c-correlation-approaches-circulant

> Within a mouse, the two-point correlation function of the tuning curves approaches a circulant
> form as more neurons are included, as does the centered and variance-normalized correlation
> function, whose eigenvectors converge to Fourier modes, as circular symmetry predicts.

In [ ]:
top4 = [int(np.where(names == n)[0][0]) for n in ["A3707", "A3716", "A3711", "A3706"]]
sizes = [5, 10, 20, 40, 80]
def circ_dists(rng):
    out = {}
    for m in top4:
        tcs = Phi_data[mouse_nums == m]
        for kind in ("raw", "norm"):
            row = []
            for n in sizes:
                ds = []
                for _ in range(100):
                    sub = tcs[rng.choice(len(tcs), n, replace=False)]
                    if kind == "norm":
                        sub = (sub - sub.mean(0)) / sub.std(0)
                    ds.append(H.rel_dist_from_circulant(sub.T @ sub / n))
                row.append(np.mean(ds))
            out[(names[m], kind)] = row
    return out
def falls(row):
    return row[-1] < row[0] and int((np.diff(row) > 0).sum()) <= 1
res3c = {}
for k in range(5):
    sd = H.seed("fig3c-correlation-approaches-circulant") + 1000 * k
    res3c[sd] = circ_dists(np.random.default_rng(sd))
sd0 = H.seed("fig3c-correlation-approaches-circulant")
tab3c = pd.DataFrame({f"{a} {b}": np.round(v, 4) for (a, b), v in res3c[sd0].items()}, index=[f"N={n}" for n in sizes]).T
print("mean relative distance from circulant form, 100 subsets per size, claim seed:")
print(tab3c.to_string())
part1 = {s_: all(falls(v) for v in r_.values()) for s_, r_ in res3c.items()}
print("part 1 (distance falls, 80 < 5, at most one increase) for all mice and both functions, per seed:", part1)
print("distance at N=80 over five seeds, range:", {k_: (round(min(r_[k_][-1] for r_ in res3c.values()), 4), round(max(r_[k_][-1] for r_ in res3c.values()), 4)) for k_ in res3c[sd0]})
part2 = True
for m in top4:
    tcs = Phi_data[mouse_nums == m]; c = (tcs - tcs.mean(0)) / tcs.std(0); C = c.T @ c / len(c)
    w, V = np.linalg.eigh(C); V = V[:, ::-1]
    pur = [H.fourier_purity(V[:, j]) for j in range(5)]
    ok = all(p >= 0.8 for p, _ in pur[:4]) and pur[0][1] == 1 and pur[1][1] == 1
    part2 &= ok
    print(f"{names[m]} (N={len(tcs)}): leading eigenvectors (power fraction, frequency):", [(round(p, 3), f) for p, f in pur], "| holds:", ok)
print("part 2 holds for all four mice:", part2)

**Result: reproduced.** For all four mice and both functions the distance from circulant form
falls at every step from 5 to 80 neurons (for example A3707, 0.73 to 0.27 raw and 0.71 to 0.16
centered and normalized), under all five seeds. The leading four eigenvectors of each mouse's
centered and normalized function have 89% to 98% of their power at one frequency, eigenvectors 1
and 2 at frequency 1 and 3 and 4 at frequency 2.

## fig4-generative-fit

> A single parameter set of the generative process, fitted to the two-point correlation
> function pooled over mice, is (sigma, beta, b) of about (1.42, 2.77, 1.73).

The grid is the criterion's (`ring/gp_opt.py`: sigma in linspace(1, 2, 20), b in linspace(0.01,
5, 30), beta in linspace(0.01, 10, 30); error over the first 20 coefficients). The criterion
asks for 500,000 samples per evaluation; on this runner's CPUs that is about 3 hours for the
18,000 points. So the whole grid is first evaluated with 25,000 samples, the same draws for every
point (common random numbers, float32 as the paper's torch run on a GPU), and the 30 best points
are then evaluated again with 500,000 fresh samples each, which decide the minimum. This is a
deviation, recorded in the attempt; the shortlist pass and the final pass are both shown.

In [ ]:
_, Gdata_ft = H.calc_C(Phi_data)
scale_vals = np.linspace(1, 2, 20); bias_vals = np.linspace(0.01, 5, 30); beta_vals = np.linspace(0.01, 10, 30)
def grid_pass(rng, N=25000):
    z = H.sample_z(rng, N, 51, dtype=np.float32).astype(np.complex64)
    L = np.zeros((20, 30, 30), dtype=np.float64)
    for i, sc in enumerate(scale_vals):
        X = np.fft.irfft(z * np.sqrt(H.wrapped_gaussian_ft(sc)).astype(np.float32)[None, :], axis=1, norm="forward").astype(np.float32)
        for j, b_ in enumerate(bias_vals):
            Xb = X - np.float32(b_)
            for k, be in enumerate(beta_vals):
                P = H.softplus(Xb, np.float32(be)); P = P / (P.mean(1, keepdims=True) + 1e-30)
                C = np.mean(np.abs(np.fft.rfft(P, axis=1, norm="forward")[:, :20]) ** 2, 0)
                L[i, j, k] = np.sqrt(((C - Gdata_ft[:20]) ** 2).sum())
    return L
def final_pass(L, rng, n_best=30, N=500000):
    idx = np.argsort(L.ravel())[:n_best]
    out = []
    for f in idx:
        i, j, k = np.unravel_index(f, L.shape)
        errs = []
        for chunk in range(5):   # 5 x 100,000 = 500,000 samples, in chunks to bound memory
            P = H.gen_Phi_m(scale_vals[i], bias_vals[j], beta_vals[k], 100000, rng)
            errs.append(np.mean(np.abs(np.fft.rfft(P, axis=1, norm="forward")[:, :20]) ** 2, 0))
        C = np.mean(errs, 0)
        out.append((np.sqrt(((C - Gdata_ft[:20]) ** 2).sum()), scale_vals[i], bias_vals[j], beta_vals[k], L[i, j, k]))
    return sorted(out)
t = time.time()
sd0 = H.seed("fig4-generative-fit")
L = grid_pass(np.random.default_rng(sd0))
i, j, k = np.unravel_index(L.argmin(), L.shape)
print(f"shortlist pass (25,000 samples) in {time.time()-t:.0f} s: minimum at sigma {scale_vals[i]:.4f}, b {bias_vals[j]:.4f}, beta {beta_vals[k]:.4f}, error {L.min():.5f}")
fits = {}
for kk in range(5):
    sd = sd0 + 1000 * kk
    fp = final_pass(L, np.random.default_rng(sd + 1))   # second generator of the claim: seed + 1
    fits[sd] = fp[0]
    if kk == 0:
        fp0 = fp
    print(f"seed {sd}: final pass (500,000 samples) best sigma {fp[0][1]:.4f}, b {fp[0][2]:.4f}, beta {fp[0][3]:.4f}, error {fp[0][0]:.5f}")
print("final pass, claim seed, the ten best of the 30:")
print(pd.DataFrame(fp0[:10], columns=["error_500k", "sigma", "b", "beta", "error_25k"]).round(4).to_string(index=False))
FIT = (fits[sd0][1], fits[sd0][2], fits[sd0][3])
print(f"fitted (sigma, beta, b) = ({FIT[0]:.4f}, {FIT[2]:.4f}, {FIT[1]:.4f}); expected (1.42 +- 0.142, 2.77 +- 0.345, 1.73 +- 0.173)")
print("over five seeds: sigma", sorted({round(v[1], 4) for v in fits.values()}), "beta", sorted({round(v[3], 4) for v in fits.values()}), "b", sorted({round(v[2], 4) for v in fits.values()}))
print("grid point nearest the paper's parameters: sigma", scale_vals[8].round(4), "b", bias_vals[10].round(4), "beta", beta_vals[8].round(4),
      "| shortlist error there", L[8, 10, 8].round(5), "| rank of that point in the shortlist pass:", int((L < L[8, 10, 8]).sum()) + 1)
# variant: the two best points again, with 500,000 samples drawn and transformed in float32, as the
# paper's torch run on a GPU computes them
def err32(params, rng):
    Cs = []
    for chunk in range(5):
        z = H.sample_z(rng, 100000, 51, dtype=np.float32).astype(np.complex64)
        X = np.fft.irfft(z * np.sqrt(H.wrapped_gaussian_ft(params[0])).astype(np.float32)[None, :], axis=1, norm="forward").astype(np.float32)
        P = H.softplus(X - np.float32(params[1]), np.float32(params[2])); P = P / (P.mean(1, keepdims=True) + 1e-30)
        Cs.append(np.mean(np.abs(np.fft.rfft(P, axis=1, norm="forward")[:, :20]) ** 2, 0))
    return np.sqrt(((np.mean(Cs, 0) - Gdata_ft[:20]) ** 2).sum())
best_pt = (fits[sd0][1], fits[sd0][2], fits[sd0][3])
paper_pt = (scale_vals[8], bias_vals[10], beta_vals[8])
for kk in range(5):
    rng32 = np.random.default_rng(sd0 + 2 + 1000 * kk)   # third generator of the claim: seed + 2
    print(f"float32 variant, seed {sd0 + 1000*kk}: error at the best point {err32(best_pt, rng32):.5f}, at the paper's grid point {err32(paper_pt, rng32):.5f}")
print("minimum error over b and beta at each sigma (shortlist pass):", dict(zip(scale_vals.round(3), L.min((1, 2)).round(5))))
print(f"time {time.time()-t:.0f} s")

**Result: partially reproduced.** With the final pass of 500,000 samples the minimum is at sigma
1.4737, b 1.2145, beta 3.1103 under all five seeds: sigma (1.42 within 0.142) and beta (2.77
within 0.345) match and b (1.73 within 0.173) does not. The paper's grid point (1.4211, 1.7307,
2.7659) is the minimum of the 25,000-sample shortlist pass and the second best in the final pass
(error 0.0095 against 0.0089); in the float32 variant it is the better of the two under the
claim's seed (0.00902 against 0.00920) and the worse under the other four. The error surface has
two near-equal minima, and which one wins depends on the draws and the arithmetic.

## Generative samples for Fig. 5

100,000 samples, each divided by its mean, at the parameters fitted above (the claims depend on
fig4-generative-fit), and as a variant at the paper's parameters. One set per claim, from each
claim's seed; four more seeds for the seed check.

In [ ]:
def gen(params, claim, k=0):
    return H.gen_Phi_m(*params, 100000, np.random.default_rng(H.seed(claim) + 1000 * k))
print("fitted parameters (scale, bias, beta):", np.round(FIT, 4), "| paper's:", PAPER_PARAMS)

## fig5ab-profiles-match

> The center-of-mass-aligned mean and fluctuation profiles predicted by the fitted generative
> process match those of the data.

In [ ]:
def prof_match(G):
    c = H.center_tuning_curves(G); gm, gs = c.mean(0), c.std(0)
    out = {}
    for lab, g, ms_, avg in (("mean", gm, mus, mu_avg), ("sd", gs, sds, sd_avg)):
        within = ((g >= ms_.min(0)) & (g <= ms_.max(0))).mean()
        peak_rel = abs(g.max() - avg.max()) / avg.max()
        out[lab] = dict(within=within, peak_rel=peak_rel, a=within >= 0.9, b=peak_rel <= 0.1, gpeak=g.max(), dpeak=avg.max())
    return out
for lab, params in (("fitted", FIT), ("paper", PAPER_PARAMS)):
    for k in range(5):
        o = prof_match(gen(params, "fig5ab-profiles-match", k))
        print(f"{lab} params, seed +{1000*k}: " + "; ".join(f"{p}: within range {v['within']:.2f}, peak {v['gpeak']:.3f} vs {v['dpeak']:.3f} (rel {v['peak_rel']:.3f}), (a) {v['a']}, (b) {v['b']}" for p, v in o.items()))

**Result: reproduced.** At the fitted parameters the generative mean profile lies within the range
of the 31 mouse profiles at all 100 offsets and the standard deviation profile at 97, and their
peaks are within 3.0% and 1.3% of the mouse-average peaks; both readings hold for both profiles
under all five seeds. At the paper's parameters the same holds (within range at 100 offsets, peaks
within 0.2% and 1.9%).

## fig5c-peak-counts-match

> The generative process reproduces the distribution of peak counts per tuning curve across
> detection thresholds, with the strongest agreement at the higher thresholds.

In [ ]:
for lab, params in (("fitted", FIT), ("paper", PAPER_PARAMS)):
    for k in range(5):
        G = gen(params, "fig5c-peak-counts-match", k)
        line = []
        for z in (1.0, 0.5, 0.0):
            p, qq = H.peak_count_hist(Phi_data, z), H.peak_count_hist(G, z)
            line.append(f"z {z}: TV {H.tv_distance(p, qq):.3f}, Hellinger {H.hellinger(p, qq):.3f}")
            if k == 0:
                print(f"  {lab} z {z}: data {np.round(p, 3)}, generative {np.round(qq, 3)}")
        print(f"{lab} params, seed +{1000*k}: " + "; ".join(line))

**Result: reproduced.** At zthresh 1 the total variation distance between the data and generative
peak-count histograms is 0.022 and the Hellinger distance 0.061, the smallest of the three
thresholds (0.065 and 0.086 at 0.5, 0.128 and 0.124 at 0), and below 0.15, under all five seeds.
At the paper's parameters the order is the same (0.026, 0.040, 0.105).

## fig5d-peak-heights-match

> The generative process reproduces the full distributions of tuning-curve values at the three
> highest peaks.

In [ ]:
pv_data = H.get_peak_values(Phi_data, 3)
for lab, params in (("fitted", FIT), ("paper", PAPER_PARAMS)):
    for k in range(5):
        pvg = H.get_peak_values(gen(params, "fig5d-peak-heights-match", k), 3)
        line = []
        for j in range(3):
            a_, b_ = pv_data[:, j][~np.isnan(pv_data[:, j])], pvg[:, j][~np.isnan(pvg[:, j])]
            rel = abs(np.median(b_) - np.median(a_)) / np.median(a_)
            line.append(f"peak {j+1}: n {len(a_)}/{len(b_)}, median {np.median(a_):.3f}/{np.median(b_):.3f} (rel {rel:.3f}), KS {H.ks_distance(a_, b_):.3f}")
        print(f"{lab} params, seed +{1000*k}: " + "; ".join(line))

**Result: reproduced.** Under reading (a) the median generative heights of the three highest peaks
are within 0.8%, 1.5% and 5.3% of the data medians (5.63, 2.90 and 2.26). Under reading (b) the
Kolmogorov-Smirnov distance is 0.058 and 0.054 for the first two peaks and 0.186 for the third (55
data curves), so (b) holds for two of three. Both hold the same way under the five seeds and at
the paper's parameters (third peak 8.8% and KS 0.253).

## fig5e-flip-symmetry-match

> The generative process captures the distribution of flip-symmetry correlations, particularly
> its strongly asymmetric tail.

In [ ]:
sym_data = H.symmetry_scores(Phi_data)
print(f"data: median {np.median(sym_data):.3f}; fraction below 0.5 {np.mean(sym_data < 0.5):.4f} ({(sym_data < 0.5).sum()} curves); below 0 {np.mean(sym_data < 0):.4f} ({(sym_data < 0).sum()} curves)")
for lab, params in (("fitted", FIT), ("paper", PAPER_PARAMS)):
    for k in range(5):
        sg = H.symmetry_scores(gen(params, "fig5e-flip-symmetry-match", k))
        fa, fb = np.mean(sg < 0.5), np.mean(sg < 0)
        print(f"{lab} params, seed +{1000*k}: median {np.median(sg):.3f}; below 0.5 {fa:.4f} (ratio to data {fa/np.mean(sym_data < 0.5):.2f}); below 0 {fb:.4f} (ratio {fb/max(np.mean(sym_data < 0), 1e-12):.2f})")

**Result: reproduced.** 23.7% of data curves have a flip-symmetry correlation below 0.5 and 10.3%
below 0; the generative process gives 28.1% and 11.8%, ratios of 1.19 and 1.14, within the factor
1.5 under both readings and all five seeds. The medians are 0.927 (data) and 0.834 (generative).

## fig5f-information-match

> The generative process captures the distribution of head-direction information.

In [ ]:
inf_data = H.bits_per_spike(Phi_data)
print(f"data: median {np.median(inf_data):.4f}, range {inf_data.min():.4f} to {inf_data.max():.4f}")
for lab, params in (("fitted", FIT), ("paper", PAPER_PARAMS)):
    for k in range(5):
        ig = H.bits_per_spike(gen(params, "fig5f-information-match", k))
        print(f"{lab} params, seed +{1000*k}: median {np.median(ig):.4f} (rel {abs(np.median(ig)-np.median(inf_data))/np.median(inf_data):.3f}), KS {H.ks_distance(inf_data, ig):.3f}")

**Result: reproduced.** The median head-direction information is 1.288 for the data and 1.368 for
the generative process, 6.2% apart, and the Kolmogorov-Smirnov distance is 0.138, below 0.15; both
readings hold under all five seeds, and at the paper's parameters (4.3%, 0.109).

## fig2l-recorded-bump-tracks-hd

> The overlap order parameter computed directly from the recordings, with the measured tuning
> curves and binned spike counts of one mouse, forms a bump that tracks the animal's head
> direction.

As `closing_the_loop.ipynb`: the session with most head-direction cells (A3707), 10-ms counts
from the first head-direction sample of the square epoch, averaged in blocks of 10 bins (100 ms);
each cell's counts and curve are divided by the mean of its curve; m is the mean over cells of
curve times counts. The angle of the peak of m is the centre of its bin (a standard choice; the
bins are floor(hd / (2 pi / 100))). The head direction of a 100-ms bin is the circular mean of
its samples; bins with any untracked sample are left out of the comparison.

In [ ]:
d = sessions[int(np.where(names == "A3707")[0][0])]
s = d["sess"]
hd_idx = d["hd_idx"]; N_bins = len(hd_idx)
t_hd = s["hd_t"]; i0 = np.argmin((t_hd - d["ti"]) ** 2)
hd_raw = s["hd"][i0:i0 + N_bins]
counts = np.stack([H.spike_counts(st, d["ti"], d["tf"], N_bins) for st in s["spikes"]], 1)  # (T, 117)
def bump_track(tc_full, counts, hd_raw, sel=None):
    fac = 1.0 / (tc_full.mean(1) + 1e-12)
    Xc = tc_full * fac[:, None]
    nb = len(counts) // 10
    cc = counts[:nb * 10].reshape(nb, 10, -1).mean(1) * fac[None, :]
    m = Xc.T @ cc.T / len(fac)                       # (100, nb)
    peak = (m.argmax(0) + 0.5) * 2 * np.pi / 100
    hh = hd_raw[:nb * 10].reshape(nb, 10)
    ok = ~np.isnan(hh).any(1)
    if sel is not None:
        ok &= sel
    hm = np.angle(np.exp(1j * hh[ok]).mean(1)) % (2 * np.pi)
    err = H.circ_diff(peak[ok], hm)
    return np.median(err), circcorrcoef(peak[ok], hm), ok.sum(), nb, m, err
med, rc, nok, nb, m_rec, err = bump_track(d["opt"].mean(1), counts, hd_raw)
print(f"A3707: {counts.shape[1]} cells, {nb} bins of 100 ms, {nok} with tracked head direction")
print(f"median |peak of m - head direction| = {med:.3f} rad (pi/8 = {np.pi/8:.3f}); circular correlation {rc:.3f} (>= 0.9)")
print("error percentiles 25, 50, 75, 90:", np.percentile(err, [25, 50, 75, 90]).round(3))
# variant: curves of one partition applied to the other partition's spikes
mask = H.alternating_mask(N_bins)
bmask = mask[:nb * 10].reshape(nb, 10)
for part, sel in ((0, ~bmask.any(1)), (1, bmask.all(1))):
    med_v, rc_v, nok_v, _, _, _ = bump_track(d["opt"][:, part], counts, hd_raw, sel)
    print(f"variant: curves of partition {part} on the other partition's bins ({nok_v}): median error {med_v:.3f} rad, circular r {rc_v:.3f}")
# null: head direction shifted by 60 s against m
sh = 600
pk = (m_rec.argmax(0) + 0.5) * 2 * np.pi / 100
hb = np.angle(np.exp(1j * hd_raw[:nb * 10].reshape(nb, 10)).mean(1))
okn = ~np.isnan(hb[sh:])
med_n = np.median(H.circ_diff(pk[:-sh][okn], hb[sh:][okn]))
print(f"reference: with the head direction shifted by 60 s, median error {med_n:.3f} rad (uniform chance pi/2 = {np.pi/2:.3f})")
fig, ax = plt.subplots(figsize=(8, 1.6))
ax.imshow(m_rec[:, :900], aspect="auto", origin="lower", extent=(0, 90, 0, 2 * np.pi), vmax=np.percentile(m_rec, 97.5))
hh = np.angle(np.exp(1j * hd_raw[:9000].reshape(900, 10)).mean(1)) % (2 * np.pi)
ax.plot(np.arange(900) * 0.1, hh, "r.", ms=1); ax.set_xlabel("time (s)"); ax.set_ylabel("theta")
plt.show()
print("first 90 s shown; m range", m_rec.min().round(3), m_rec.max().round(3))

**Result: reproduced.** In A3707, over 17,990 bins of 100 ms with tracked head direction, the
median absolute difference between the angle of the peak of m and the head direction is 0.168 rad,
below pi/8 (0.393), against 1.294 rad with the head direction shifted by 60 s. The circular
correlation is 0.896, just below the 0.9 of reading (b), so the claim holds under reading (a).
With curves from one partition applied to the other partition's spikes the median error is 0.172
and 0.177 rad.

## Summary

| claim | outcome |
|---|---|
| hd-cell-selection (link) | reproduced |
| tuning-curves-match-cache (link, with inputs) | reproduced |
| fig1e-heterogeneous-peaks | reproduced |
| fig5d-peak-numbers | reproduced |
| fig1fg-mean-and-fluctuation-profiles | reproduced |
| fig3ab-centers-of-mass-uniform | reproduced |
| fig3c-correlation-approaches-circulant | reproduced |
| fig4-generative-fit | partially reproduced |
| fig5ab-profiles-match | reproduced |
| fig5c-peak-counts-match | reproduced |
| fig5d-peak-heights-match | reproduced |
| fig5e-flip-symmetry-match | reproduced |
| fig5f-information-match | reproduced |
| fig2l-recorded-bump-tracks-hd | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method. The generative fit evaluates the whole grid with 25,000
samples and only the 30 best points with 500,000, where the paper uses 500,000 at every point;
random draws come from each claim's seed, not numpy seed 42; the Poisson likelihood that picks
the smoothing width drops a term that does not depend on the curve.

The dataset. At version 0.240528.1542 the files hold what the paper describes: 31 sessions,
2,691 units, 1,533 head-direction units (all also labelled excitatory), the wake_square epoch in
every file, head direction in radians at 10 ms, and nine files with an optogenetic epoch. The
tuning curves rebuilt from them match the paper's cached curves to 1e-14 Hz. Spikes and head
direction share a clock: in every session the head-direction information peaks within 0.05 s of
lag 0. No recording appears twice.

In [ ]:
print(f"total time {time.time()-T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")